# Scaled Dot-Product Attention from Scratch

This notebook builds the central attention calculation with plain PyTorch tensor operations. The goal is conceptual transparency, not a production Transformer implementation.

By the end, we will be able to explain:

- why learned projections create queries, keys, and values;
- why dot products are divided by $\sqrt{d_k}$;
- why softmax is applied across keys;
- how causal masking prevents a decoder from seeing future tokens; and
- how attention output preserves the sequence shape.

## 1. Token representations

A tokenizer would normally map text to token IDs, and an embedding table would map those IDs to vectors. Tokenization is the subject of Stage 2, so here we begin with a small matrix $X$. Each row represents one token and each column one feature.

In [1]:
import math

import torch

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)

tokens = ["The", "model", "attends", "causally"]
sequence_length = len(tokens)
d_model = 8
d_k = 4
d_v = 4

X = torch.randn(sequence_length, d_model)

print(f"PyTorch: {torch.__version__}")
print(f"Tokens: {tokens}")
print(f"X shape: {tuple(X.shape)} = (sequence length, model dimension)")
print(X)

/Users/aleynakilic/Documents/ChatGPT/ai/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch: 2.13.0
Tokens: ['The', 'model', 'attends', 'causally']
X shape: (4, 8) = (sequence length, model dimension)
tensor([[-0.820,  0.396,  0.899, -1.388, -0.167,  0.285, -0.641, -0.894],
        [ 0.927, -0.536, -1.160, -0.460,  0.709,  1.013,  0.230,  1.090],
        [-1.583, -0.325,  1.926, -0.330,  0.198,  0.782,  1.039, -0.725],
        [-0.209, -0.215, -1.816, -0.345, -2.061,  0.674, -1.323, -1.360]])


## 2. Create queries, keys, and values

The same input has three different jobs:

- **Query ($Q$):** what information is this token looking for?
- **Key ($K$):** what information does this token advertise?
- **Value ($V$):** what content should be passed forward if this token receives attention?

Learned matrices let the model represent these jobs in different spaces. During training, gradient descent learns the matrices; here they are seeded random values so the calculation is reproducible.

In [2]:
W_q = torch.randn(d_model, d_k) / math.sqrt(d_model)
W_k = torch.randn(d_model, d_k) / math.sqrt(d_model)
W_v = torch.randn(d_model, d_v) / math.sqrt(d_model)

Q = X @ W_q
K = X @ W_k
V = X @ W_v

assert Q.shape == (sequence_length, d_k)
assert K.shape == (sequence_length, d_k)
assert V.shape == (sequence_length, d_v)

print(f"Q shape: {tuple(Q.shape)}")
print(f"K shape: {tuple(K.shape)}")
print(f"V shape: {tuple(V.shape)}")
print("Q:\n", Q)

Q shape: (4, 4)
K shape: (4, 4)
V shape: (4, 4)
Q:
 tensor([[-1.177,  0.478,  0.290, -0.858],
        [-0.125, -0.966, -0.827,  1.616],
        [-1.516,  0.789,  0.900, -2.703],
        [-1.023, -0.631, -0.752,  1.649]])


## 3. Score every query against every key

The matrix product $QK^T$ produces one score for every query-key pair. A large positive dot product means their learned directions align. The result has shape `sequence length × sequence length`.

As $d_k$ grows, the variance of an unscaled dot product grows too. Large magnitudes can make softmax extremely peaked, where gradients become small. Dividing by $\sqrt{d_k}$ stabilizes the score scale.

In [3]:
raw_scores = Q @ K.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(d_k)

unscaled_weights = torch.softmax(raw_scores, dim=-1)
attention_weights = torch.softmax(scaled_scores, dim=-1)
unmasked_output = attention_weights @ V

assert raw_scores.shape == (sequence_length, sequence_length)
assert torch.allclose(attention_weights.sum(dim=-1), torch.ones(sequence_length))
assert unmasked_output.shape == (sequence_length, d_v)

print("Raw attention scores:\n", raw_scores)
print("\nScaled scores:\n", scaled_scores)
print("\nNormalized attention weights:\n", attention_weights)
print("\nEach row sums to:", attention_weights.sum(dim=-1))
print("\nUnmasked output shape:", tuple(unmasked_output.shape))
print(
    "Mean largest probability, unscaled vs scaled:",
    round(unscaled_weights.max(dim=-1).values.mean().item(), 3),
    round(attention_weights.max(dim=-1).values.mean().item(), 3),
)

Raw attention scores:
 tensor([[ 0.852, -0.818,  1.606, -3.244],
        [-2.645,  1.727, -2.146, -0.537],
        [ 3.397, -2.829,  3.354, -2.931],
        [-2.936,  1.934, -1.328, -3.653]])

Scaled scores:
 tensor([[ 0.426, -0.409,  0.803, -1.622],
        [-1.323,  0.864, -1.073, -0.268],
        [ 1.698, -1.414,  1.677, -1.466],
        [-1.468,  0.967, -0.664, -1.826]])

Normalized attention weights:
 tensor([[0.331, 0.144, 0.483, 0.043],
        [0.071, 0.633, 0.091, 0.204],
        [0.484, 0.022, 0.474, 0.020],
        [0.065, 0.744, 0.146, 0.046]])

Each row sums to: tensor([1.000, 1.000, 1.000, 1.000])

Unmasked output shape: (4, 4)
Mean largest probability, unscaled vs scaled: 0.745 0.586


### Why softmax?

Raw dot products are unbounded and may be negative. Softmax converts each query's scores into non-negative weights that sum to one, which makes the output a weighted mixture of value vectors. It also remains differentiable for training.

## 4. Add a causal mask

A decoder-only language model predicts the next token from tokens already available. During training, the whole sequence exists in memory, so positions above the diagonal must be hidden. We replace those future-token scores with negative infinity before softmax; their probabilities then become exactly zero.

In [4]:
causal_mask = torch.triu(
    torch.ones(sequence_length, sequence_length, dtype=torch.bool),
    diagonal=1,
)
masked_scores = scaled_scores.masked_fill(causal_mask, float("-inf"))
masked_attention_weights = torch.softmax(masked_scores, dim=-1)
masked_output = masked_attention_weights @ V

assert torch.all(masked_attention_weights[causal_mask] == 0)
assert torch.allclose(masked_attention_weights.sum(dim=-1), torch.ones(sequence_length))
assert torch.allclose(masked_attention_weights[-1], attention_weights[-1])
assert masked_output.shape == (sequence_length, d_v)

print("Causal mask (True means blocked):\n", causal_mask)
print("\nMasked scores:\n", masked_scores)
print("\nMasked attention weights:\n", masked_attention_weights)
print("\nMasked output:\n", masked_output)

Causal mask (True means blocked):
 tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

Masked scores:
 tensor([[ 0.426,   -inf,   -inf,   -inf],
        [-1.323,  0.864,   -inf,   -inf],
        [ 1.698, -1.414,  1.677,   -inf],
        [-1.468,  0.967, -0.664, -1.826]])

Masked attention weights:
 tensor([[1.000, 0.000, 0.000, 0.000],
        [0.101, 0.899, 0.000, 0.000],
        [0.494, 0.022, 0.484, 0.000],
        [0.065, 0.744, 0.146, 0.046]])

Masked output:
 tensor([[ 0.375, -0.044, -0.763,  1.090],
        [-1.009, -0.370,  1.040, -0.125],
        [ 0.188,  0.438, -1.183,  1.479],
        [-0.788, -0.255,  0.660,  0.084]])


The first token can now attend only to itself. The final token's row is unchanged because it has no future positions to hide. This triangular dependency structure is what makes the operation causal.

## 5. Demonstrate that masking blocks future information

We will drastically change all embeddings after the first token. Without a mask, the first output changes because it attends to future keys and values. With a causal mask, the first output remains identical.

In [5]:
def run_attention(inputs: torch.Tensor, *, causal: bool) -> torch.Tensor:
    """Apply this notebook's single-head attention calculation."""

    queries = inputs @ W_q
    keys = inputs @ W_k
    values = inputs @ W_v
    scores = (queries @ keys.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        mask = torch.triu(torch.ones(len(inputs), len(inputs), dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ values


changed_future = X.clone()
changed_future[1:] += 10.0

unmasked_before = run_attention(X, causal=False)
unmasked_after = run_attention(changed_future, causal=False)
masked_before = run_attention(X, causal=True)
masked_after = run_attention(changed_future, causal=True)

unmasked_first_change = torch.linalg.vector_norm(unmasked_before[0] - unmasked_after[0]).item()
masked_first_change = torch.linalg.vector_norm(masked_before[0] - masked_after[0]).item()

assert unmasked_first_change > 0
assert torch.allclose(masked_before[0], masked_after[0])

print(f"First-output change without mask: {unmasked_first_change:.6f}")
print(f"First-output change with mask:    {masked_first_change:.6f}")

First-output change without mask: 17.756536
First-output change with mask:    0.000000


## 6. From one head to a Transformer block

This notebook implements one attention head. **Multi-head attention** repeats the calculation with separate learned projections, concatenates the head outputs, and projects them back to the model dimension. Different heads can learn different relationships.

A Transformer block also surrounds attention and its feed-forward network with **residual connections** and normalization. A residual connection adds a sublayer's input back to its output, helping information and gradients travel through deep networks. Those components are important, but implementing a full block would distract from this stage's attention objective.

## Acceptance recap

- $Q$, $K$, and $V$ give each token separate lookup, matching, and content roles.
- Division by $\sqrt{d_k}$ controls dot-product variance before softmax.
- Softmax produces differentiable, normalized mixing weights.
- Causal masking sets future-token probabilities to zero.
- The output contains one value-sized vector per input token.

Questions to answer without the notebook:

1. Which axis does softmax use here, and why?
2. Why is the causal mask applied before softmax?
3. Why does the final query row have the same weights with and without the mask?
4. What would change when using multiple heads?